# Fixed-Point Iterations and Cobweb Diagrams

A **fixed point** of a function $f : \mathbb{R} \to \mathbb{R}$ is a point $x^*$ with $f(x^*) = x^*$. Many numerical algorithms, physical equilibria, and dynamical systems reduce to finding such points. The most direct strategy is **Picard iteration**:
$$
x_{n+1} = f(x_n), \quad n = 0, 1, 2, \ldots
$$
starting from an initial guess $x_0$. Whether this converges — and how fast — depends critically on the behavior of $f$ near $x^*$.

## Banach fixed-point theorem

If $f$ is a **contraction** on a complete metric space $(X, d)$, i.e. there exists $L < 1$ such that $d(f(x), f(y)) \leq L \cdot d(x,y)$ for all $x, y$, then:
1. $f$ has a **unique** fixed point $x^*$.
2. The iteration $x_{n+1} = f(x_n)$ converges to $x^*$ for **any** starting point $x_0$.
3. The convergence is **geometric** (linear): $|x_n - x^*| \leq \frac{L^n}{1-L} |x_1 - x_0|$.

For a differentiable $f$, the local contraction condition near $x^*$ is simply $|f'(x^*)| < 1$.

## Stability classification

Given a fixed point $x^*$ of $f$:
- **Attractive** ($|f'(x^*)| < 1$): nearby orbits converge to $x^*$. The smaller $|f'(x^*)|$, the faster the convergence.
- **Repulsive** ($|f'(x^*)| > 1$): nearby orbits diverge. The iteration $x_{n+1} = f(x_n)$ started near $x^*$ will move away.
- **Neutral** ($|f'(x^*)| = 1$): the linearization is inconclusive; higher-order terms determine behavior.

## Cobweb diagram

A **cobweb diagram** (or Lamrey diagram) visualizes the orbit $(x_0, x_1, x_2, \ldots)$ on the plane by drawing:
- The graph $y = f(x)$.
- The diagonal $y = x$ (whose intersections with $y = f(x)$ are the fixed points).
- A zigzag path: from $(x_n, x_n)$ go vertically to $(x_n, f(x_n)) = (x_n, x_{n+1})$, then horizontally to $(x_{n+1}, x_{n+1})$.

For an attracting fixed point the cobweb spirals inward; for a repelling one it spirals outward.

## Environment

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Dropdown

plt.rcParams["figure.dpi"] = 120

## The function family $f_\eta(x) = x + \eta \sin(2\pi x)$

We study the one-parameter family
$$
f_\eta(x) = x + \eta \sin(2\pi x).
$$
The fixed points satisfy $f_\eta(x^*) = x^*$, i.e. $\eta \sin(2\pi x^*) = 0$, so $x^* \in \mathbb{Z}$. The derivative is
$$
f_\eta'(x) = 1 + 2\pi\eta \cos(2\pi x).
$$
At the integer fixed points: $f_\eta'(n) = 1 + 2\pi\eta$. So:
- If $|1 + 2\pi\eta| < 1$, i.e. $-\frac{1}{\pi} < \eta < 0$, the fixed points are **attracting**.
- If $\eta > 0$, then $|1 + 2\pi\eta| > 1$ and the integer fixed points are **repelling** (but half-integer points $x^* = n + \tfrac{1}{2}$ become attracting when $|1 - 2\pi\eta| < 1$).

In [ ]:
def f_eta(x, eta):
    return x + eta * np.sin(2 * np.pi * x)


def df_eta(x, eta):
    return 1 + 2 * np.pi * eta * np.cos(2 * np.pi * x)


def cobweb_orbit(f, x0, n_steps=30):
    """Generate cobweb path as a list of (x, y) pairs."""
    xs = [x0, x0]
    ys = [x0, f(x0)]
    for _ in range(n_steps - 1):
        xn = ys[-1]
        xs.append(xn)
        ys.append(xn)
        xs.append(xn)
        ys.append(f(xn))
    return np.array(xs), np.array(ys)


print("Fixed-point tools ready.")

## Cobweb diagrams: convergence and divergence

We show four representative parameter values of $\eta$:
- $\eta = -0.31$: attracting integer fixed points — the cobweb spirals in.
- $\eta = -0.5$: stronger attraction.
- $\eta = +0.15$: repelling integer fixed points, attracting half-integers — the cobweb converges to $x^* = 0.5$.
- $\eta = +0.32$: repelling orbit with richer dynamics.

In [ ]:
x_plot = np.linspace(-0.3, 1.3, 800)

configs = [
    (-0.31, 0.05),
    (-0.50, 0.05),
    (+0.15, 0.05),
    (+0.32, 1.15),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, (eta, x0) in zip(axes.ravel(), configs):
    f = lambda x, e=eta: f_eta(x, e)   # noqa
    fp_deriv = df_eta(0.0, eta)
    stability = "attractive" if abs(fp_deriv) < 1 else "repulsive"

    ax.plot(x_plot, x_plot, 'k-', lw=1.5, label="$y = x$")
    ax.plot(x_plot, f(x_plot), 'b-', lw=2.5, label="$y = f_\\eta(x)$")

    # Multiple starting points
    starts = [x0 - 0.05, x0, x0 + 0.07]
    for i, x0_i in enumerate(starts):
        xs, ys = cobweb_orbit(f, x0_i, n_steps=25)
        t = i / (len(starts) - 1)
        col = (t, 0.0, 1 - t)
        ax.plot(xs, ys, '-', lw=1.2, color=col, alpha=0.8)
        ax.plot(x0_i, x0_i, 's', ms=7, color=col)

    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(fr"$\eta = {eta}$,  $f'(0) = {fp_deriv:.3f}$  ({stability})", fontsize=9)
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle(r"Cobweb diagrams for $f_\eta(x) = x + \eta\sin(2\pi x)$", y=1.02, fontsize=12)
plt.tight_layout()
plt.show()

## Derivative map and the stability condition

The stability of the orbit depends on $|f'(x)|$ along the orbit. Where $|f'(x)| < 1$, the function contracts; where $|f'(x)| > 1$, it expands. This plot shows $|f'(x)|$ for several values of $\eta$ and marks the unit line.

In [ ]:
eta_list = [-0.5, -0.3, -0.15, 0.0, 0.15, 0.32]
colors_d = plt.cm.coolwarm(np.linspace(0, 1, len(eta_list)))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.axhline(1.0, color='k', lw=1.5, ls='--', label="$|f'| = 1$ (neutral)")
ax.fill_between(x_plot, 0, 1, alpha=0.07, color='green', label="contraction ($<1$)")

for eta, col in zip(eta_list, colors_d):
    deriv = np.abs(df_eta(x_plot, eta))
    ax.plot(x_plot, deriv, lw=2, color=col, label=fr"$\eta={eta}$")

ax.set_xlim(-0.3, 1.3); ax.set_ylim(0, 3.2)
ax.set_xlabel("$x$"); ax.set_ylabel("$|f_\\eta'(x)|$")
ax.set_title(r"Derivative magnitude: contraction ($< 1$) vs expansion ($> 1$)")
ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Convergence rate

When iteration converges, the error $e_n = |x_n - x^*|$ decays geometrically:
$$
e_{n+1} \approx |f'(x^*)| \cdot e_n, \quad \text{so} \quad e_n \approx e_0 \cdot |f'(x^*)|^n.
$$
Plotting $\log e_n$ versus $n$ reveals a straight line with slope $\log|f'(x^*)|$. Faster convergence corresponds to $|f'(x^*)|$ closer to $0$.

In [ ]:
# Attracting fixed point at 0 when eta < 0
eta_conv = [-0.10, -0.15, -0.25, -0.40]
x0_start = 0.05
n_iter = 40

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for eta in eta_conv:
    f_loc = lambda x, e=eta: f_eta(x, e)
    x_star = 0.0  # attracting fixed point for eta < 0
    errs = [abs(x0_start)]
    xn = x0_start
    for _ in range(n_iter):
        xn = f_loc(xn)
        errs.append(abs(xn - x_star))
        if errs[-1] < 1e-14:
            break
    errs = np.array(errs)
    slope = abs(df_eta(0.0, eta))
    theory = errs[0] * slope ** np.arange(len(errs))
    axes[0].semilogy(errs, '-o', ms=4, lw=2, label=fr"$\eta={eta}$")
    axes[1].semilogy(theory, '--', lw=1.5,
                     label=fr"$|f'(0)|={slope:.3f}$")

axes[0].set_xlabel("iteration $n$"); axes[0].set_ylabel("$|x_n - x^*|$")
axes[0].set_title("Convergence (observed)"); axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel("iteration $n$"); axes[1].set_ylabel("$|f'(x^*)|^n \cdot e_0$")
axes[1].set_title("Theoretical linear rate"); axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)

plt.suptitle(r"Geometric convergence: $e_n \approx |f'(x^*)|^n \cdot e_0$", y=1.02)
plt.tight_layout()
plt.show()

## Interactive cobweb

Vary $\eta$ and the starting point $x_0$ to explore convergent, divergent, and oscillatory orbits. The color of the cobweb path transitions from blue (early) to red (late), so you can see the direction of travel.

In [ ]:
def show_cobweb(eta=-0.31, x0=0.7, n_steps=30):
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize

    f = lambda x, e=eta: f_eta(x, e)   # noqa
    xp = np.linspace(-0.3, 1.3, 800)

    xs, ys = cobweb_orbit(f, x0, n_steps)

    # Build colored line segments for cobweb
    points = np.array([xs, ys]).T.reshape(-1, 1, 2)
    segs = np.concatenate([points[:-1], points[1:]], axis=1)
    norm = Normalize(vmin=0, vmax=len(segs))
    lc = LineCollection(segs, cmap='RdYlBu_r', norm=norm)
    lc.set_array(np.arange(len(segs)))
    lc.set_linewidth(1.8)

    fp_deriv = df_eta(0.0, eta)
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.plot(xp, xp, 'k-', lw=1.5, alpha=0.7)
    ax.plot(xp, f(xp), 'navy', lw=2.5)
    ax.add_collection(lc)
    ax.plot(x0, x0, 'go', ms=10, zorder=5, label=f"$x_0 = {x0:.2f}$")
    ax.plot(xs[-1], xs[-1], 'r*', ms=14, zorder=5, label=f"$x_{{{n_steps}}} = {xs[-1]:.4f}$")
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(fr"$\eta = {eta:.2f}$,  $f'(0) = {fp_deriv:.3f}$", fontsize=12)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

interact(
    show_cobweb,
    eta=FloatSlider(value=-0.31, min=-0.5, max=0.5, step=0.01, description="$\\eta$"),
    x0=FloatSlider(value=0.7, min=-0.2, max=1.2, step=0.05, description="$x_0$"),
    n_steps=__import__('ipywidgets').IntSlider(value=30, min=5, max=80, step=5,
                                              description="steps"),
);

## Bibliographical resources

- Banach, S. (1922). Sur les opérations dans les ensembles abstraits et leur application aux équations intégrales. *Fundamenta Mathematicae*, 3, 133–181.
- Picard, É. (1890). Mémoire sur la théorie des équations aux dérivées partielles et la méthode des approximations successives. *Journal de Mathématiques Pures et Appliquées*, 6, 145–210.
- Devaney, R. L. (1989). *An Introduction to Chaotic Dynamical Systems* (2nd ed.). Westview Press.
- Strogatz, S. H. (1994). *Nonlinear Dynamics and Chaos*. Addison-Wesley.
- Kelley, C. T. (1995). *Iterative Methods for Linear and Nonlinear Equations*. SIAM.